# 03 · Nested spatial cross-validation (the long run)

**Two feature sets** (`settings.toml → [features]`):
* **XYZ** = Easting, Northing, top elevation of the cell;
* **XYZ_VAHDW** = XYZ + the **VA-HDW features of Li et al. (2026)**: for every reference borehole, the horizontal distance **d**, the class at the cell's elevation in that borehole **Φ** (vertical alignment) and the influence coefficient **S**. The reference boreholes are always the **training boreholes of that fit**: the outer test boreholes are never references, and every inner-CV model builds its features from its own inner-training boreholes. A cell's own borehole is masked.

```
for each outer fold k = 1..5 (blockCV folds):
    test  = boreholes in fold k
    train = all other boreholes (with a purge: minus those closer than the purge distance)
    inner CV = the other 4 outer folds
        Optuna tries hyperparameter sets and the weight of filled cells;
        each set is scored by the inner Macro-F1 at its best threshold
    refit the best set on all training boreholes -> predict fold k
```

| Block size | Purge | Models |
|---|---|---|
| 1× (main, 106 m) | none | RF, XGB, GWRF, GWXGB on XYZ **and** on XYZ_VAHDW; **GBARNN** on XYZ_VAHDW; Li et al.'s comparison models **KNN** and **SVM** on XYZ (normalised, grid search over their Table 6 grid) |
| 2× and 3× | none | XGB on both feature sets |
| 1× | 1× and 2× the block size | XGB on both feature sets |

**Run time:** measured per model in the README. Every outer fold is saved when it finishes; if the PC restarts, run the notebook again and it continues. To get the main results first, set `RUN_ONLY = ["_x1_purge0"]`.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib
from tqdm import tqdm          # text progress bars: work in every Jupyter, nothing to install

RUN_ONLY = None        # None = all runs; or parts of run names, e.g. ["_x1_purge0"] or ["GBARNN"]

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds = {m: H.load_folds(S, m) for m in S["blockcv"]["multiples"]}
block_m = folds[1][1]
runs = H.run_list(S)
if RUN_ONLY:
    runs = [r for r in runs if any(s in r["tag"] for s in RUN_ONLY)]
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["blockcv"]["k"] + 1))
PRIV = H.out_dir(S, "03_spatial_cv", private=True)
(PRIV / "checkpoints").mkdir(exist_ok=True); (PRIV / "models").mkdir(exist_ok=True)
print(f"Block size {block_m:.1f} m | {len(runs)} runs x {len(outer)} outer folds | {len(cells)} cells, "
      f"{cells.BH_Code.nunique()} boreholes")
display(pd.DataFrame(runs))

## Run (or continue)
One progress bar per run; it moves one step per Optuna trial. Next to it: the outer folds finished so far, with the **best** and **mean** Macro-F1 and accuracy on their test boreholes.

In [ ]:
def fold_score(pred):
    return H.macro_f1(pred.y, pred.y_pred), float((pred.y == pred.y_pred).mean())

for r in runs:
    fold_ids, block = folds[r["multiple"]]
    purge_m = r["purge_x"] * block_m
    key = H.settings_key(S, cells, fold_ids) + f"_{purge_m:.3f}"
    per_fold = H.n_trials(r["model"], S)
    done = {}
    for k in outer:
        ck = PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib"
        if ck.exists():
            saved = joblib.load(ck)
            if saved["key"] != key:
                raise RuntimeError(f"{ck.name} was made with other data or settings: delete the folder "
                                   f"{PRIV / 'checkpoints'} (and models) to start again.")
            done[k] = fold_score(saved["predictions"])
    bar = tqdm(total=len(outer) * per_fold, initial=len(done) * per_fold, desc=f"{r['tag']:<28}",
               bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} trials [{elapsed}<{remaining}]{postfix}")

    def show():
        if done:
            f1 = [v[0] for v in done.values()]; acc = [v[1] for v in done.values()]
            best = max(done, key=lambda k: done[k][0])
            bar.set_postfix_str(f"folds {len(done)}/{len(outer)} | best fold {best}: F1 {done[best][0]:.3f} "
                                f"acc {done[best][1]:.3f} | mean F1 {np.mean(f1):.3f} acc {np.mean(acc):.3f}")
    show()
    for k in outer:
        if k in done:
            continue
        res = H.run_fold(r, cells, fold_ids, k, purge_m, S, on_trial=lambda: bar.update(1))
        res["predictions"] = res["predictions"].assign(model=r["model"], features=r["features"], multiple=r["multiple"],
                                                       block_m=block, purge_x=r["purge_x"], purge_m=purge_m)
        res["info"].update(model=r["model"], features=r["features"], multiple=r["multiple"], block_m=block, purge_m=purge_m)
        if r["multiple"] == 1 and r["purge_x"] == 0:       # keep the fold models of the main runs for notebook 06
            joblib.dump(res["model"], PRIV / "models" / f"{r['tag']}_fold{k}.joblib", compress=3)
        res.pop("model")
        joblib.dump({**res, "key": key}, PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib", compress=3)
        done[k] = fold_score(res["predictions"])
        show()
    bar.close()
print(f"All requested folds done ({(time.time() - T0) / 3600:.2f} h in this session).")

## Collect the results

In [ ]:
preds, infos, trials = [], [], []
for f in sorted((PRIV / "checkpoints").glob("*.joblib")):
    res = joblib.load(f)
    preds.append(res["predictions"]); infos.append(res["info"])
    trials.append(res["trials"].assign(tag=res["info"]["tag"], outer_fold=res["info"]["outer_fold"]))
preds = pd.concat(preds, ignore_index=True)
info = pd.DataFrame(infos)
preds.to_csv(PRIV / "outer_predictions.csv", index=False)
pd.concat(trials, ignore_index=True).to_csv(PRIV / "optuna_trials.csv", index=False)
OUT = H.out_dir(S, "03_spatial_cv")
info.drop(columns=["params"]).to_csv(OUT / "fold_info.csv", index=False)
done_t = info.groupby("tag").agg(folds=("outer_fold", "count"), hours=("seconds", lambda s: s.sum() / 3600))
display(done_t.round(2))
print(f"{len(preds)} test predictions saved. Next: notebook 04.")